# AgriVision - CSA47 Deep Learning Assessment 1

This Kaggle-oriented notebook consolidates the preprocessing, mathematical verification, PCA, baseline, capacity, bootstrap, optimisation, performance and 38-class design-challenge workflow.

**Before running:** attach the PlantVillage dataset in Kaggle. The code expects `/kaggle/input/plantvillage-dataset/plantvillage dataset/color` or previously saved arrays under `/kaggle/working/agriVision`. Run cells in order. Some model-training cells take several minutes, and time measurements vary between sessions.

Captured outputs from the completed Kaggle run are included in the accompanying PDF report. The notebook cells are provided so you can re-run and check the analysis. Review the convergence warnings and deployment limitations in the report before using the results.

In [ ]:
# CSA47 Deep Learning - CO1 Assessment 1 (AgriVision)
# Consolidated reproducibility code for Kaggle Notebooks.
#
# Prerequisites:
# 1. Attach the PlantVillage dataset in Kaggle.
# 2. Run the dataset-manifest, split and preprocessing cells from the workflow,
#    or create the required arrays in /kaggle/working/agriVision.
# 3. Set OUT to the directory containing those arrays.
#
# Outputs reported in the accompanying assessment report are captured from the
# user's Kaggle execution. Timing values may vary on rerun and should not be
# expected to match exactly across notebook sessions or hardware.

from pathlib import Path
from time import perf_counter, perf_counter_ns
import json
import gc
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, f1_score
)
from sklearn.exceptions import ConvergenceWarning

OUT = Path('/kaggle/working/agriVision')
OUT.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
RANDOM_STATE = 42

DATASET MANIFEST AND PREPROCESSING
This cell is needed if the manifests / processed arrays do not already exist.

In [ ]:
DATA_ROOT = Path('/kaggle/input/plantvillage-dataset/plantvillage dataset')
COLOR_DIR = DATA_ROOT / 'color'

if COLOR_DIR.is_dir():
    records = []
    for class_dir in sorted(COLOR_DIR.iterdir()):
        if not class_dir.is_dir():
            continue
        for image_path in class_dir.iterdir():
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append({
                    'path': str(image_path),
                    'label': class_dir.name,
                    'crop_group': 'tomato' if class_dir.name.startswith('Tomato___') else 'other'
                })
    manifest = pd.DataFrame(records)
    if manifest.empty:
        raise ValueError('No color image files were found.')
    manifest.to_csv(OUT / 'manifest_all38.csv', index=False)
    tomato_df = manifest[manifest['crop_group'] == 'tomato'].reset_index(drop=True)
    all38_df = manifest.reset_index(drop=True)
    print('Color dataset classes:', manifest['label'].nunique())
    print('Color dataset images:', len(manifest))
    print('Tomato images:', len(tomato_df))
else:
    print('Color source folder not found. Reuse existing saved arrays if available.')

# 70/15/15 stratified splits. This is an experimental choice, not a prescribed
# split in the assessment.
def make_splits(frame, seed=RANDOM_STATE):
    train_val, test = train_test_split(
        frame, test_size=0.15, random_state=seed, stratify=frame['label']
    )
    train, val = train_test_split(
        train_val, test_size=(0.15 / 0.85), random_state=seed + 1,
        stratify=train_val['label']
    )
    return tuple(x.reset_index(drop=True) for x in (train, val, test))


def preprocess_frame(frame, label_to_id, split_name, size=(64, 64)):
    from PIL import Image
    from tqdm.auto import tqdm
    X = np.empty((len(frame), size[0] * size[1]), dtype=np.uint8)
    y = np.empty(len(frame), dtype=np.int64)
    for i, row in enumerate(tqdm(frame.itertuples(index=False), total=len(frame), desc=split_name)):
        try:
            with Image.open(row.path) as im:
                gray = im.convert('L').resize(size, Image.Resampling.LANCZOS)
                X[i] = np.asarray(gray, dtype=np.uint8).reshape(-1)
            y[i] = label_to_id[row.label]
        except Exception as exc:
            raise RuntimeError(f'Failed image: {row.path}') from exc
    return X, y

# If the arrays already exist, do not rebuild unnecessarily.
required_tomato = [
    'X_train_uint8.npy', 'y_train.npy', 'X_val_uint8.npy', 'y_val.npy',
    'X_test_uint8.npy', 'y_test.npy', 'tomato_class_names.json'
]
if not all((OUT / f).is_file() for f in required_tomato) and 'tomato_df' in globals():
    tomato_train, tomato_val, tomato_test = make_splits(tomato_df)
    tomato_class_names = sorted(tomato_df['label'].unique())
    tomato_label_to_id = {name: i for i, name in enumerate(tomato_class_names)}
    for split_name, frame in [('train', tomato_train), ('val', tomato_val), ('test', tomato_test)]:
        X, y = preprocess_frame(frame, tomato_label_to_id, f'tomato_{split_name}')
        np.save(OUT / f'X_{split_name}_uint8.npy', X)
        np.save(OUT / f'y_{split_name}.npy', y)
    with open(OUT / 'tomato_class_names.json', 'w', encoding='utf-8') as f:
        json.dump(tomato_class_names, f, indent=2)

# Build 38-class arrays when they are not already saved.
required_38 = [
    'X38_train_uint8.npy', 'y38_train.npy', 'X38_val_uint8.npy',
    'y38_val.npy', 'X38_test_uint8.npy', 'y38_test.npy',
    'all38_class_names.json'
]
if not all((OUT / f).is_file() for f in required_38) and 'all38_df' in globals():
    all38_train, all38_val, all38_test = make_splits(all38_df)
    class_names_38 = sorted(all38_df['label'].unique())
    label_to_id_38 = {name: i for i, name in enumerate(class_names_38)}
    for split_name, frame in [('train', all38_train), ('val', all38_val), ('test', all38_test)]:
        X, y = preprocess_frame(frame, label_to_id_38, f'all38_{split_name}')
        np.save(OUT / f'X38_{split_name}_uint8.npy', X)
        np.save(OUT / f'y38_{split_name}.npy', y)
    with open(OUT / 'all38_class_names.json', 'w', encoding='utf-8') as f:
        json.dump(class_names_38, f, indent=2)

TASK 1 - REPRESENTATION AND NORMS

In [ ]:
X_train_u8 = np.load(OUT / 'X_train_uint8.npy')
y_train = np.load(OUT / 'y_train.npy')
y_val = np.load(OUT / 'y_val.npy')
y_test = np.load(OUT / 'y_test.npy')
with open(OUT / 'tomato_class_names.json', encoding='utf-8') as f:
    tomato_class_names = json.load(f)

x = X_train_u8[0].astype(np.float64) / 255.0
L1 = np.sum(np.abs(x))
L2 = np.sqrt(np.sum(x ** 2))
Linf = np.max(np.abs(x))
assert np.isclose(L1, np.linalg.norm(x, ord=1))
assert np.isclose(L2, np.linalg.norm(x, ord=2))
assert np.isclose(Linf, np.linalg.norm(x, ord=np.inf))
print('TASK 1 NORMS')
print('Features:', len(x), 'L1:', f'{L1:.4f}', 'L2:', f'{L2:.4f}', 'Linf:', f'{Linf:.4f}')
print('Shapes: image=(64,64), vector=(4096,), batch=(32,4096), training=', X_train_u8.shape)

TASK 2 - PCA / EIGENVALUES / RECONSTRUCTION

In [ ]:
X_train = X_train_u8.astype(np.float32) / 255.0
X_val_u8 = np.load(OUT / 'X_val_uint8.npy')
X_val = X_val_u8.astype(np.float32) / 255.0

pca_file = OUT / 'tomato_pca_256.joblib'
if pca_file.exists():
    pca = joblib.load(pca_file)
else:
    pca = PCA(n_components=256, svd_solver='randomized', random_state=42, iterated_power=4)
    t0 = perf_counter(); pca.fit(X_train); print('PCA fit seconds:', f'{perf_counter()-t0:.4f}')
    joblib.dump(pca, pca_file)

singular_values = pca.singular_values_
eigenvalues = singular_values ** 2 / (len(X_train) - 1)
cumulative = np.cumsum(pca.explained_variance_ratio_)
print('First ten singular values:', np.round(singular_values[:10], 4))
print('First ten covariance eigenvalues:', np.round(eigenvalues[:10], 4))

ks = np.array([5, 10, 20, 50, 100, 150, 200, 256])
rng = np.random.default_rng(42)
ids = rng.choice(len(X_val), size=min(200, len(X_val)), replace=False)
X_eval = X_val[ids]
Xc = X_eval - pca.mean_
Z = pca.transform(X_eval)
recon_rows = []
for k in ks:
    reconstructed = Z[:, :k] @ pca.components_[:k]
    err = np.linalg.norm(Xc - reconstructed, ord='fro') / np.linalg.norm(Xc, ord='fro')
    recon_rows.append({'k': int(k), 'explained_variance_pct': 100 * cumulative[k-1], 'relative_reconstruction_error': err})
recon_df = pd.DataFrame(recon_rows)
recon_df.to_csv(OUT / 'pca_reconstruction_results.csv', index=False)
print(recon_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 4.2)); ax.plot(ks, 100*cumulative[ks-1], marker='o')
ax.set_title('Cumulative Explained Variance vs PCA Components'); ax.set_xlabel('Components k'); ax.set_ylabel('Explained variance (%)'); ax.grid(True, alpha=.3); fig.tight_layout(); fig.savefig(OUT/'pca_explained_variance.png', dpi=180); plt.show()
fig, ax = plt.subplots(figsize=(7.5, 4.2)); ax.plot(ks, recon_df['relative_reconstruction_error'], marker='o')
ax.set_title('Validation Reconstruction Error vs PCA Components'); ax.set_xlabel('Components k'); ax.set_ylabel('Relative Frobenius error'); ax.grid(True, alpha=.3); fig.tight_layout(); fig.savefig(OUT/'pca_reconstruction_error.png', dpi=180); plt.show()

# Visualise leading principal components as 64 x 64 coefficient images.
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for pc, ax in enumerate(axes.flat):
    ax.imshow(pca.components_[pc].reshape(64, 64), cmap='coolwarm')
    ax.set_title(f'PC {pc+1}'); ax.axis('off')
fig.suptitle('Leading PCA Components of Tomato Leaf Images')
fig.tight_layout(); fig.savefig(OUT/'pca_component_images.png', dpi=180); plt.show()

# Measure PCA fitting cost at the required dataset scales for 64 components.
pca_timing_rows = []
for sample_n in sorted(set([1000, 5000, 10000, len(X_train)])):
    if sample_n == len(X_train): X_sub = X_train
    else: X_sub = X_train[np.random.default_rng(42+sample_n).choice(len(X_train), size=sample_n, replace=False)]
    timed_pca = PCA(n_components=64, svd_solver='randomized', random_state=42, iterated_power=3)
    t0 = perf_counter(); timed_pca.fit(X_sub); elapsed = perf_counter()-t0
    pca_timing_rows.append({'Training_images': sample_n, 'Components': 64, 'Fit_time_seconds': elapsed,
                            'Explained_variance_pct': 100*np.sum(timed_pca.explained_variance_ratio_)})
pca_time_df = pd.DataFrame(pca_timing_rows)
pca_time_df.to_csv(OUT/'pca_fit_time_by_dataset_size.csv', index=False)
print('PCA fitting times by sample size:')
print(pca_time_df.round(4).to_string(index=False))

# A real 2x2 covariance subproblem; eigenvalues and vectors are checked by NumPy.
variances = np.var(X_train, axis=0)
i = int(np.argmax(variances)); ci = X_train[:, i] - X_train[:, i].mean()
covs = ((X_train - X_train.mean(axis=0)).T @ ci) / (len(X_train) - 1); covs[i] = 0
j = int(np.argmax(np.abs(covs)))
C = np.cov(X_train[:, [i, j]], rowvar=False, ddof=1)
vals, Q = np.linalg.eigh(C); order = np.argsort(vals)[::-1]; vals = vals[order]; Q = Q[:, order]
print('2x2 covariance matrix:\n', C)
print('Eigenvalues:', np.round(vals, 4)); print('Eigenvectors (columns):\n', np.round(Q, 4))
print('Reconstruction Frobenius error:', f'{np.linalg.norm(C-Q@np.diag(vals)@Q.T, ord="fro"):.10f}')

TASKS 3 AND 5 - BASELINE MODELS AND INFORMATION THEORY

In [ ]:
from sklearn.metrics import log_loss

def build_model(name):
    if name == 'SGD_log_loss':
        return SGDClassifier(loss='log_loss', alpha=0.0001, max_iter=100, tol=1e-3, random_state=42)
    if name == 'KNN_k5':
        return KNeighborsClassifier(n_neighbors=5, weights='distance', algorithm='brute', n_jobs=-1)
    if name == 'DecisionTree':
        return DecisionTreeClassifier(max_depth=20, min_samples_leaf=2, random_state=42)
    raise ValueError(name)

def info_metrics(y_true, probs, n_classes):
    eps = 1e-12
    p = np.bincount(y_true, minlength=n_classes).astype(np.float64) / len(y_true)
    q = np.clip(probs.mean(axis=0), eps, None); q /= q.sum()
    mask = p > 0
    H = -np.sum(p[mask] * np.log2(p[mask]))
    Hpq = -np.sum(p[mask] * np.log2(q[mask]))
    KLpq = np.sum(p[mask] * np.log2(p[mask] / q[mask]))
    KLqp = np.sum(q[mask] * np.log2(q[mask] / p[mask]))
    return {'H_true_bits': H, 'marginal_cross_entropy_bits': Hpq,
            'KL_p_to_q_bits': KLpq, 'KL_q_to_p_bits': KLqp,
            'identity_gap': abs(Hpq - (H + KLpq))}

# Uses PCA-100 and raw representations, matching the original experiment.
Z_train100 = pca.transform(X_train)[:, :100].astype(np.float32)
Z_val100 = pca.transform(X_val)[:, :100].astype(np.float32)
results = []
for rep, (Xa, Xv) in {
    'Raw_4096': (X_train, X_val),
    'PCA_100': (Z_train100, Z_val100)
}.items():
    for name in ['SGD_log_loss', 'KNN_k5', 'DecisionTree']:
        model = build_model(name); t0 = perf_counter()
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always', ConvergenceWarning)
            model.fit(Xa, y_train)
        fit_s = perf_counter() - t0
        t0 = perf_counter(); preds = model.predict(Xv); predict_s = perf_counter() - t0
        accuracy = accuracy_score(y_val, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(y_val, preds, average='macro', zero_division=0)
        row = {'Model': name, 'Representation': rep, 'Accuracy': accuracy, 'Macro_precision': prec, 'Macro_recall': rec, 'Macro_F1': f1, 'Training_time_seconds': fit_s, 'Validation_prediction_seconds': predict_s}
        if hasattr(model, 'predict_proba'):
            probs = model.predict_proba(Xv)
            aligned = np.zeros((len(Xv), len(tomato_class_names)), dtype=float)
            for idx, cls in enumerate(model.classes_): aligned[:, int(cls)] = probs[:, idx]
            row.update(info_metrics(y_val, aligned, len(tomato_class_names)))
        results.append(row)
        joblib.dump(model, OUT / f'{name}_{rep}.joblib')
        print(name, rep, 'accuracy=', f'{accuracy:.4f}', 'macroF1=', f'{f1:.4f}', 'fit_s=', f'{fit_s:.4f}', 'predict_s=', f'{predict_s:.4f}')
        del model; gc.collect()
baselines = pd.DataFrame(results)
baselines.to_csv(OUT / 'baseline_reproduction_results.csv', index=False)
print(baselines.round(4).to_string(index=False))

TASK 4 - NUMERICAL STABILITY AND CONDITIONING

In [ ]:
z = np.array([1000.0, 999.0, 995.0])
with np.errstate(over='ignore', invalid='ignore', divide='ignore'):
    naive = np.exp(z) / np.exp(z).sum()
m = np.max(z); stable_exp = np.exp(z-m); stable = stable_exp / stable_exp.sum()
lse = m + np.log(stable_exp.sum())
print('Naive softmax:', naive)
print('Stable probabilities:', np.round(stable, 4), 'sum=', f'{stable.sum():.4f}')
print('Stable log-sum-exp:', f'{lse:.4f}')
assert np.all(np.isfinite(stable)) and np.isclose(stable.sum(), 1)
condition_number = pca.explained_variance_[0] / pca.explained_variance_[-1]
print('Condition number retained PCA-256 covariance subspace:', f'{condition_number:.4f}')

TASK 6 - CAPACITY CURVE (also maps to the extra Part (e) plot instruction)

In [ ]:
cap_rows = []
capacities = {'C1': 2, 'C2': 4, 'C3': 8, 'C4': 16, 'C5': 32}
for label, depth in capacities.items():
    model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=2, random_state=42)
    t0 = perf_counter(); model.fit(Z_train100, y_train); fit_s = perf_counter() - t0
    tr_error = 1 - accuracy_score(y_train, model.predict(Z_train100))
    va_error = 1 - accuracy_score(y_val, model.predict(Z_val100))
    cap_rows.append({'Capacity': label, 'Depth': depth, 'Training_error': tr_error,
                     'Validation_error': va_error, 'Generalisation_gap': va_error-tr_error, 'Fit_time': fit_s})
cap_df = pd.DataFrame(cap_rows)
cap_df.to_csv(OUT / 'capacity_error_results.csv', index=False)
print(cap_df.round(4).to_string(index=False))
best_idx = int(np.argmin(cap_df['Validation_error'].to_numpy()))
fig, ax = plt.subplots(figsize=(8, 5))
ax.axvspan(-.5, 1.5, alpha=.10, label='Underfitting region (C1-C2)')
ax.axvspan(2.5, 4.5, alpha=.08, label='Overfitting region (C4-C5)')
ax.plot(range(5), cap_df['Training_error'], marker='o', label='Training error')
ax.plot(range(5), cap_df['Validation_error'], marker='s', label='Validation error')
ax.scatter([best_idx], [cap_df.iloc[best_idx]['Validation_error']], marker='*', s=170, label=f"Optimal tested capacity: {cap_df.iloc[best_idx]['Capacity']}")
ax.vlines(best_idx, cap_df.iloc[best_idx]['Training_error'], cap_df.iloc[best_idx]['Validation_error'], linestyles='dashed', label=f"Generalisation gap = {cap_df.iloc[best_idx]['Generalisation_gap']:.4f}")
ax.set_xticks(range(5), [f"{c}\n(depth={d})" for c,d in zip(cap_df['Capacity'],cap_df['Depth'])])
ax.set_xlabel('Model capacity'); ax.set_ylabel('Classification error rate'); ax.set_title('Training and Validation Error vs Model Capacity (C1-C5)')
ax.grid(True, alpha=.3); ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT/'capacity_training_validation_error.png', dpi=200); plt.show()

TASK 7 - BOOTSTRAP BIAS / VARIANCE AND MLE / MAP

In [ ]:
Z_train = Z_train100; Z_val = Z_val100
n_classes = len(tomato_class_names); Y_val = np.eye(n_classes)[y_val]
rng = np.random.default_rng(42); B = 10; bv_rows = []
for name, depth in [('Low capacity (depth 2)', 2), ('High capacity (depth 32)', 32)]:
    prediction_replicas = []
    for b in range(B):
        idx = rng.integers(0, len(Z_train), size=len(Z_train))
        model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=2, random_state=1000+b)
        model.fit(Z_train[idx], y_train[idx]); raw = model.predict_proba(Z_val)
        aligned = np.zeros((len(Z_val), n_classes))
        for col, cls in enumerate(model.classes_): aligned[:, int(cls)] = raw[:, col]
        prediction_replicas.append(aligned)
    P = np.stack(prediction_replicas)
    meanP = P.mean(axis=0)
    bias2 = np.mean(np.sum((meanP-Y_val)**2, axis=1))
    variance = np.mean(np.sum(np.var(P, axis=0, ddof=0), axis=1))
    brier = np.mean(np.sum((P-Y_val[None,:,:])**2, axis=2))
    bv_rows.append({'Model':name, 'Replicas':B, 'Bias_squared_proxy':bias2, 'Variance':variance, 'Mean_Brier_loss':brier, 'Residual':brier-bias2-variance})
print(pd.DataFrame(bv_rows).round(4).to_string(index=False))

# Beta(2,2) prior for a class proportion
with open(OUT/'tomato_class_names.json', encoding='utf-8') as f: names = json.load(f)
target_id = names.index('Tomato___Tomato_Yellow_Leaf_Curl_Virus')
rows=[]; rng = np.random.default_rng(2026)
for n in sorted(set([100,500,1000,5000,len(y_train)])):
    subset = y_train if n == len(y_train) else y_train[rng.choice(len(y_train),size=n,replace=False)]
    s = int(np.sum(subset == target_id)); mle=s/n; mapv=(s+1)/(n+2)
    rows.append({'n':n,'s':s,'MLE':mle,'MAP_Beta_2_2':mapv,'Absolute_difference':abs(mle-mapv)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

TASK 8 - SOFTMAX REGRESSION GRADIENTS AND OPTIMISERS

In [ ]:
# Loss: J = -mean(log(p_true)) + (lambda/2)*||W||_F^2
# Gradients: dW = X.T @ (P-Y)/N + lambda*W; db = mean(P-Y).
def stable_softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    expz = np.exp(shifted)
    return expz / expz.sum(axis=1, keepdims=True)

def loss_values(X, y, W, b, reg=1e-4, batch_size=1024):
    total = np.zeros(W.shape[0], dtype=float)
    n_total = len(y)
    for start in range(0, n_total, batch_size):
        end = min(start + batch_size, n_total)
        xb, yb = X[start:end], y[start:end]
        for r in range(W.shape[0]):
            logits = xb @ W[r] + b[r]
            m = logits.max(axis=1, keepdims=True)
            lse = m[:, 0] + np.log(np.exp(logits-m).sum(axis=1))
            total[r] += np.sum(lse - logits[np.arange(len(yb)), yb])
    return total/n_total + 0.5*reg*np.sum(W*W, axis=(1,2))

# Standardise PCA coordinates using train-only statistics, as in the experiment.
scaler100 = StandardScaler()
Xs_train = scaler100.fit_transform(Z_train100).astype(np.float32)
Xs_val = scaler100.transform(Z_val100).astype(np.float32)
rates = np.array([0.001, 0.01, 0.1]); epochs = 8; batch_size = 128; reg = 1e-4
rng = np.random.default_rng(42)
optimizer_summary = []; optimizer_histories = {}; optimizer_times = []

for strategy in ['batch', 'sgd', 'mini_batch']:
    W = np.zeros((len(rates), Xs_train.shape[1], len(tomato_class_names)), dtype=np.float64)
    b = np.zeros((len(rates), len(tomato_class_names)), dtype=np.float64)
    hist = {'train': [], 'val': []}; t0 = perf_counter()
    for epoch in range(epochs):
        perm = rng.permutation(len(Xs_train))
        if strategy == 'batch': batches = [np.arange(len(Xs_train))]
        elif strategy == 'mini_batch': batches = [perm[i:i+batch_size] for i in range(0,len(perm),batch_size)]
        else: batches = [perm[i:i+1] for i in range(len(perm))]
        for idx in batches:
            xb, yb = Xs_train[idx], y_train[idx]; bn = len(yb)
            logits = np.einsum('nd,rdc->rnc', xb, W) + b[:,None,:]
            P = stable_softmax(logits.reshape(-1, logits.shape[-1])).reshape(logits.shape)
            E = P.copy()
            E[np.arange(len(rates))[:,None], np.arange(bn)[None,:], yb[None,:]] -= 1.0
            gradW = np.einsum('nd,rnc->rdc', xb, E)/bn + reg*W
            gradb = E.mean(axis=1)
            W -= rates[:,None,None]*gradW
            b -= rates[:,None]*gradb
            if not np.isfinite(W).all() or not np.isfinite(b).all():
                raise FloatingPointError(f'Non-finite parameters for {strategy}')
        hist['train'].append(loss_values(Xs_train, y_train, W, b, reg))
        hist['val'].append(loss_values(Xs_val, y_val, W, b, reg))
    elapsed = perf_counter()-t0
    optimizer_histories[strategy] = hist
    optimizer_times.append({'Strategy':strategy,'Epochs':epochs,'Runtime_seconds':elapsed,'Runtime_per_epoch_seconds':elapsed/epochs})
    for ri, eta in enumerate(rates):
        optimizer_summary.append({'Strategy':strategy,'Learning_rate':eta,
            'Final_train_loss':hist['train'][-1][ri], 'Final_validation_loss':hist['val'][-1][ri],
            'Final_generalisation_gap':hist['val'][-1][ri]-hist['train'][-1][ri]})
    np.save(OUT/f'softmax_{strategy}_W.npy',W); np.save(OUT/f'softmax_{strategy}_b.npy',b)

opt_df = pd.DataFrame(optimizer_summary); opt_df.to_csv(OUT/'softmax_final_loss_results.csv',index=False)
time_df = pd.DataFrame(optimizer_times); time_df.to_csv(OUT/'softmax_optimizer_timing.csv',index=False)
print(opt_df.round(4).to_string(index=False)); print(time_df.round(4).to_string(index=False))

for strategy, hist in optimizer_histories.items():
    fig, ax = plt.subplots(figsize=(8, 5))
    for ri, eta in enumerate(rates):
        ax.plot(range(1,epochs+1),[h[ri] for h in hist['train']],marker='o',label=f'Train eta={eta:.3f}')
        ax.plot(range(1,epochs+1),[h[ri] for h in hist['val']],linestyle='--',label=f'Validation eta={eta:.3f}')
    ax.set_title(f'Softmax Regression Loss Curves - {strategy}'); ax.set_xlabel('Epoch')
    ax.set_ylabel('Regularised cross-entropy loss (nats)'); ax.grid(True,alpha=.3); ax.legend(fontsize=8,ncol=2)
    fig.tight_layout(); fig.savefig(OUT/f'softmax_loss_{strategy}.png',dpi=180); plt.show()

TASK 9 - PERFORMANCE AT REQUIRED DATASET SCALES

In [ ]:
from sklearn.metrics import classification_report

scales = sorted(set([1000, 5000, 10000, len(y_train)]))
model_names = ['SGD_log_loss', 'KNN_k5', 'DecisionTree']
scale_rows=[]; class_metric_rows=[]; best_confusion = None
for rep, (Xa_full, Xv) in {'Raw_4096':(X_train,X_val),'PCA_100':(Z_train100,Z_val100)}.items():
    for n in scales:
        if n == len(y_train): idx = np.arange(len(y_train))
        else:
            idx, _ = train_test_split(np.arange(len(y_train)),train_size=n,stratify=y_train,random_state=100+n)
        Xa, ya = Xa_full[idx], y_train[idx]
        for name in model_names:
            mdl=build_model(name); t0=perf_counter()
            with warnings.catch_warnings(record=True) as cw:
                warnings.simplefilter('always', ConvergenceWarning); mdl.fit(Xa,ya)
            fit_s=perf_counter()-t0
            t0=perf_counter(); pred=mdl.predict(Xv); inf_s=perf_counter()-t0
            acc=accuracy_score(y_val,pred)
            pr,rc,f1,_=precision_recall_fscore_support(y_val,pred,labels=np.arange(len(tomato_class_names)),average='macro',zero_division=0)
            counts,typ = (mdl.coef_.size+mdl.intercept_.size,'coefficients+intercepts') if hasattr(mdl,'coef_') else ((mdl.tree_.node_count,'tree nodes') if hasattr(mdl,'tree_') else (mdl._fit_X.size+len(mdl._y),'stored features+labels'))
            model_path = OUT / f'task9_{name}_{rep}_n{n}.joblib'
            joblib.dump(mdl, model_path)
            artifact_mb = model_path.stat().st_size / 1_000_000
            scale_rows.append({'Training_images':n,'Representation':rep,'Model':name,'Accuracy':acc,'Macro_precision':pr,'Macro_recall':rc,'Macro_F1':f1,'Training_time_seconds':fit_s,'Validation_inference_seconds':inf_s,'Complexity_count':counts,'Complexity_type':typ,'Serialized_model_MB':artifact_mb})
            pcr,pcrr,pcf,pcs=precision_recall_fscore_support(y_val,pred,labels=np.arange(len(tomato_class_names)),average=None,zero_division=0)
            for ci,cname in enumerate(tomato_class_names):
                class_metric_rows.append({'Training_images':n,'Representation':rep,'Model':name,'Class':cname,'Precision':pcr[ci],'Recall':pcrr[ci],'F1':pcf[ci],'Support':pcs[ci]})
            cm = confusion_matrix(y_val, pred, labels=np.arange(len(tomato_class_names)))
            cm_df = pd.DataFrame(cm, index=tomato_class_names, columns=tomato_class_names)
            cm_df.index.name = 'Actual'
            cm_df.to_csv(OUT/f'confusion_{name}_{rep}_n{n}.csv')
            if best_confusion is None or f1 > best_confusion['Macro_F1']:
                best_confusion = {'Macro_F1': f1, 'Model': name, 'Representation': rep,
                                  'Training_images': n, 'matrix': cm.copy()}
            print(n,rep,name,'accuracy=',f'{acc:.4f}','macroF1=',f'{f1:.4f}','fit=',f'{fit_s:.4f}','predict=',f'{inf_s:.4f}')
            del mdl; gc.collect()
scale_df=pd.DataFrame(scale_rows); class_df=pd.DataFrame(class_metric_rows)
scale_df.to_csv(OUT/'task9_model_performance_all_scales.csv',index=False)
class_df.to_csv(OUT/'task9_per_class_metrics.csv',index=False)
print(scale_df.round(4).to_string(index=False))
print('Best validation Macro-F1 configuration:', {k:v for k,v in best_confusion.items() if k != 'matrix'})
best_cm = best_confusion['matrix']
best_cm_df = pd.DataFrame(best_cm, index=tomato_class_names, columns=tomato_class_names)
best_cm_df.index.name = 'Actual'; best_cm_df.to_csv(OUT/'task9_best_validation_confusion.csv')
fig, ax = plt.subplots(figsize=(9,7)); im=ax.imshow(best_cm,aspect='auto'); fig.colorbar(im,ax=ax,label='Image count')
short=['Bacterial\nspot','Early\nblight','Late\nblight','Leaf\nMold','Septoria','Spider\nmites','Target\nSpot','Yellow Curl\nVirus','Mosaic\nvirus','Healthy']
ax.set_xticks(range(len(short)),short,fontsize=8); ax.set_yticks(range(len(short)),short,fontsize=8)
ax.set_xlabel('Predicted class'); ax.set_ylabel('Actual class'); ax.set_title('Validation Confusion Matrix - Best Macro-F1 configuration')
for ii in range(best_cm.shape[0]):
    for jj in range(best_cm.shape[1]): ax.text(jj,ii,str(best_cm[ii,jj]),ha='center',va='center',fontsize=7)
fig.tight_layout(); fig.savefig(OUT/'task9_confusion_matrix.png',dpi=180); plt.show()
# Most common class confusions excluding the diagonal.
conf_rows=[]
for ii in range(best_cm.shape[0]):
    for jj in range(best_cm.shape[1]):
        if ii != jj and best_cm[ii,jj] > 0:
            conf_rows.append({'Actual_class':tomato_class_names[ii], 'Predicted_class':tomato_class_names[jj], 'Misclassified_images':int(best_cm[ii,jj])})
conf_df=pd.DataFrame(conf_rows).sort_values('Misclassified_images',ascending=False)
print('Top 10 confusions:'); print(conf_df.head(10).to_string(index=False))

for metric,ylabel,title,file in [('Accuracy','Validation accuracy','Validation Accuracy vs Training Dataset Size','task9_accuracy_vs_scale.png'),('Macro_F1','Validation macro-F1','Validation Macro-F1 vs Training Dataset Size','task9_macro_f1_vs_scale.png')]:
    fig,ax=plt.subplots(figsize=(9,5))
    for rep in ['Raw_4096','PCA_100']:
        for name in model_names:
            ss=scale_df[(scale_df.Representation==rep)&(scale_df.Model==name)].sort_values('Training_images')
            ax.plot(ss.Training_images,ss[metric],marker='o',label=f'{name} | {rep}')
    ax.set_xlabel('Training images'); ax.set_ylabel(ylabel); ax.set_title(title); ax.grid(True,alpha=.3); ax.legend(fontsize=7,ncol=2)
    fig.tight_layout(); fig.savefig(OUT/file,dpi=180); plt.show()

TASK 12 - 38-CLASS LABEL NOISE AND DESIGN CHALLENGE

In [ ]:
from sklearn.preprocessing import StandardScaler

need38 = ['X38_train_uint8.npy','y38_train.npy','X38_val_uint8.npy','y38_val.npy','X38_test_uint8.npy','y38_test.npy','all38_class_names.json']
if not all((OUT/f).exists() for f in need38):
    raise FileNotFoundError('Run the 38-class preprocessing cell first.')
X38tr_u8=np.load(OUT/'X38_train_uint8.npy'); X38v_u8=np.load(OUT/'X38_val_uint8.npy'); X38t_u8=np.load(OUT/'X38_test_uint8.npy')
y38tr=np.load(OUT/'y38_train.npy').astype(int); y38v=np.load(OUT/'y38_val.npy').astype(int); y38t=np.load(OUT/'y38_test.npy').astype(int)
with open(OUT/'all38_class_names.json',encoding='utf-8') as f: names38=json.load(f)
N38=len(names38)

# Fit PCA only on training data, then fit scaling stats only on transformed train.
pca38=PCA(n_components=100,svd_solver='randomized',random_state=42,iterated_power=3)
t0=perf_counter(); Xfit=X38tr_u8.astype(np.float32)/255.0; pca38.fit(Xfit); pca_fit=perf_counter()-t0

def xform38(U,k):
    components=pca38.components_[:k].astype(np.float32); mean=pca38.mean_.astype(np.float32)
    output=np.empty((len(U),k),dtype=np.float32)
    for st in range(0,len(U),512):
        en=min(st+512,len(U)); xb=U[st:en].astype(np.float32)/255.0
        output[st:en]=(xb-mean)@components.T
    return output

Z38tr100=xform38(X38tr_u8,100); Z38v100=xform38(X38v_u8,100); Z38t100=xform38(X38t_u8,100)
scaler38=StandardScaler(); Z38tr100=scaler38.fit_transform(Z38tr100).astype(np.float32); Z38v100=scaler38.transform(Z38v100).astype(np.float32); Z38t100=scaler38.transform(Z38t100).astype(np.float32)
print('38-class PCA fit seconds:',f'{pca_fit:.4f}')
print('PCA100 cumulative explained variance:',f'{100*pca38.explained_variance_ratio_.sum():.4f}%')

# Exactly about 20% of training labels are changed. Validation/test are untouched.
rng=np.random.default_rng(42); noisy38=y38tr.copy(); m=int(round(.20*len(noisy38))); changed=rng.choice(len(noisy38),size=m,replace=False); offsets=rng.integers(1,N38,size=m); noisy38[changed]=(noisy38[changed]+offsets)%N38
noise_rate=np.mean(noisy38!=y38tr)
print('Corrupted labels:',int(np.sum(noisy38!=y38tr)),'/',len(noisy38),'noise rate:',f'{100*noise_rate:.4f}%')
np.save(OUT/'all38_train_labels_noisy20.npy',noisy38)

configs=[
('SGD_log_loss_clean_PCA100','log_loss',100,'clean'),
('SGD_log_loss_noisy20_PCA100','log_loss',100,'noisy'),
('SGD_modified_huber_noisy20_PCA100','modified_huber',100,'noisy'),
('SGD_modified_huber_noisy20_PCA64','modified_huber',64,'noisy'),
('KNN_noisy20_PCA100','knn',100,'noisy')]

def make38(kind):
    if kind=='log_loss': return SGDClassifier(loss='log_loss',alpha=1e-4,max_iter=100,tol=1e-3,random_state=42)
    if kind=='modified_huber': return SGDClassifier(loss='modified_huber',alpha=1e-4,max_iter=100,tol=1e-3,random_state=42)
    if kind=='knn': return KNeighborsClassifier(n_neighbors=5,weights='distance',algorithm='brute',n_jobs=-1)
    raise ValueError(kind)

def one_image_predict(raw_u8, k, model):
    flat=raw_u8.astype(np.float32).reshape(1,-1)/255.0
    transformed=(flat-pca38.mean_.astype(np.float32))@pca38.components_[:k].astype(np.float32).T
    scaled=(transformed-scaler38.mean_[:k].astype(np.float32))/scaler38.scale_[:k].astype(np.float32)
    return model.predict(scaled)[0]

lat_rng=np.random.default_rng(2026); lat_idx=lat_rng.choice(len(X38v_u8),size=min(100,len(X38v_u8)),replace=False)
challenge_rows=[]; saved_models={}
for name,kind,k,label_source in configs:
    Xtr=Z38tr100[:,:k]; Xv=Z38v100[:,:k]; Xt=Z38t100[:,:k]
    yfit=y38tr if label_source=='clean' else noisy38
    model=make38(kind); t0=perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always',ConvergenceWarning); model.fit(Xtr,yfit)
    fit_s=perf_counter()-t0
    warn='; '.join(str(w.message) for w in caught if issubclass(w.category,ConvergenceWarning))
    t0=perf_counter(); val_pred=model.predict(Xv); batch_predict=perf_counter()-t0
    acc=accuracy_score(y38v,val_pred); pr,rc,f1,_=precision_recall_fscore_support(y38v,val_pred,labels=np.arange(N38),average='macro',zero_division=0)
    artifact={'pca_mean':pca38.mean_.astype(np.float32),'pca_components':pca38.components_[:k].astype(np.float32),'scaler_mean':scaler38.mean_[:k].astype(np.float32),'scaler_scale':scaler38.scale_[:k].astype(np.float32),'model':model,'n_components':k,'classes':names38}
    artifact_path=OUT/f'{name}.joblib'; joblib.dump(artifact,artifact_path,compress=3); size_mb=artifact_path.stat().st_size/1_000_000
    for ii in lat_idx[:5]: one_image_predict(X38v_u8[ii],k,model)
    lats=[]
    for ii in lat_idx:
        t0=perf_counter_ns(); one_image_predict(X38v_u8[ii],k,model); lats.append((perf_counter_ns()-t0)/1_000_000)
    p50=float(np.percentile(lats,50)); p95=float(np.percentile(lats,95))
    challenge_rows.append({'Configuration':name,'Training_labels':label_source,'Loss_or_model':kind,'PCA_components':k,'Validation_accuracy':acc,'Validation_macro_precision':pr,'Validation_macro_recall':rc,'Validation_macro_F1':f1,'Classifier_fit_seconds':fit_s,'Validation_batch_prediction_seconds':batch_predict,'Pipeline_latency_mean_ms':float(np.mean(lats)),'Pipeline_latency_p50_ms':p50,'Pipeline_latency_p95_ms':p95,'Saved_pipeline_MB_decimal':size_mb,'Meets_5MB_artifact_limit':size_mb<=5.0,'Meets_50ms_Kaggle_p95_proxy':p95<=50.0,'Convergence_warning':warn})
    saved_models[name]={'model':model,'artifact':artifact,'path':artifact_path,'k':k}
    print(name,'accuracy=',f'{acc:.4f}','MacroF1=',f'{f1:.4f}','fit=',f'{fit_s:.4f}','sizeMB=',f'{size_mb:.4f}','p50ms=',f'{p50:.4f}','p95ms=',f'{p95:.4f}')
    if warn: print('Convergence warning:',warn)
    del model,artifact; gc.collect()

challenge_df=pd.DataFrame(challenge_rows); challenge_df.to_csv(OUT/'task12_38class_adaptation_results.csv',index=False)
print(challenge_df[['Configuration','Validation_accuracy','Validation_macro_F1','Pipeline_latency_p50_ms','Pipeline_latency_p95_ms','Saved_pipeline_MB_decimal','Meets_5MB_artifact_limit','Meets_50ms_Kaggle_p95_proxy']].round(4).to_string(index=False))

# Select the best noisy-label setting that meets the measured artifact and latency proxies.
eligible=challenge_df[(challenge_df.Training_labels=='noisy')&challenge_df.Meets_5MB_artifact_limit&challenge_df.Meets_50ms_Kaggle_p95_proxy]
if eligible.empty:
    print('No noisy-label configuration passed both measured proxy limits.')
else:
    chosen=eligible.sort_values('Validation_macro_F1',ascending=False).iloc[0]; name=chosen['Configuration']; mdl=saved_models[name]['model']; k=saved_models[name]['k']
    test_pred=mdl.predict(Z38t100[:,:k]); test_acc=accuracy_score(y38t,test_pred); _,_,test_f1,_=precision_recall_fscore_support(y38t,test_pred,labels=np.arange(N38),average='macro',zero_division=0)
    print('Selected:',name,'test_accuracy=',f'{test_acc:.4f}','test_macroF1=',f'{test_f1:.4f}')

# Entropy and majority-class baseline for train/validation.
for split, labels in [('Training',y38tr),('Validation',y38v)]:
    counts=np.bincount(labels,minlength=N38).astype(float); p=counts/counts.sum(); mask=p>0
    H=-np.sum(p[mask]*np.log2(p[mask])); Hmax=np.log2(N38); majority=int(np.argmax(counts))
    print(split,'entropy_bits=',f'{H:.4f}','max_entropy_bits=',f'{Hmax:.4f}','normalised_entropy=',f'{H/Hmax:.4f}','majority_class=',names38[majority],'majority_accuracy=',f'{counts[majority]/counts.sum():.4f}')

# End of consolidated reproducibility code.